In [ ]:
# ============================================
#  GPU / Device check
# ============================================
import tensorflow as tf

print("TF version:", tf.__version__)
print("Available devices:")
for d in tf.config.list_physical_devices():
    print(d)

In [ ]:
# ============================================
#  Repo setup (same as before)
# ============================================
import os

# ✅ CHANGED for HM-RNN — update your directory name and branch
baseDir = f'/mnt/c/Users/krishna/Documents/NeuroSpeech/speechBCI-{"FirstLast"}'
branch = "Krishna-Bhatia"
repo_url = "https://github.com/Aditya-Yan/NeuroSpeech.git"

if os.path.exists(os.path.join(baseDir, ".git")):
    print(f"🔁 Repo already exists at {baseDir} — pulling latest changes...")
    os.chdir(baseDir)
    !git fetch origin {branch}
    !git checkout {branch}
    !git reset --hard
    !git pull origin {branch}
else:
    print(f"📥 Cloning branch '{branch}' into {baseDir} ...")
    !git clone --branch {branch} --single-branch {repo_url} "{baseDir}"

baseDir = baseDir + '/speechBCI-main'
os.chdir(baseDir)


In [ ]:
# ============================================
#  Imports & environment setup
# ============================================
import os
from glob import glob
from pathlib import Path
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]=""   # leave empty if GPU not working

import numpy as np
from omegaconf import OmegaConf
import tensorflow as tf
from neuralDecoder.neuralSequenceDecoder import NeuralSequenceDecoder
import neuralDecoder.utils.lmDecoderUtils as lmDecoderUtils


In [ ]:
# ============================================
#  Language model loading
# ============================================
lmDir = '/mnt/c/Users/krishna/Documents/NeuroSpeech/speechBCI-main/languageModel' # do not change
ngramDecoder = lmDecoderUtils.build_lm_decoder(
    lmDir,
    acoustic_scale=0.8,
    nbest=1,
    beam=18
)


In [ ]:
# ============================================
#  Make sure local NeuralDecoder package is installed
# ============================================
!pip install -e "{baseDir}/NeuralDecoder"

import importlib
import neuralDecoder.neuralSequenceDecoder as nsd_module
importlib.reload(nsd_module)


In [ ]:
# ============================================
#  Run inference on test / competitionHoldOut
# ============================================

# ✅ CHANGED for HM-RNN — use your HM-RNN checkpoint folder
testDirs = ['test','competitionHoldOut']
trueTranscriptions = [[],[]]
decodedTranscriptions = [[],[]]

for dirIdx in range(2):
    ckptDir = baseDir + '/derived/rnns/hmrnnTrial'   # ✅ CHANGED HERE

    # ✅ HM-RNN trained model saves args.yaml as usual
    args = OmegaConf.load(os.path.join(ckptDir, 'args.yaml'))
    args['loadDir'] = ckptDir
    args['mode'] = 'infer'                           # ✅ consistent with updated code
    args['loadCheckpointIdx'] = None
    args['outputDir'] = ckptDir

    # Disable validation datasets
    for x in range(len(args['dataset']['datasetProbabilityVal'])):
        args['dataset']['datasetProbabilityVal'][x] = 0.0

    # Select sessions for testing
    for sessIdx in range(4,19):
        args['dataset']['datasetProbabilityVal'][sessIdx] = 1.0
        args['dataset']['dataDir'][sessIdx] = '/mnt/c/Users/krishna/Documents/NeuroSpeech/speechBCI-main/derived/tfRecords' # do not change
    args['testDir'] = testDirs[dirIdx]

    # Reset graph/session safely for TF2
    tf.keras.backend.clear_session()   # ✅ replaced tf.compat.v1.reset_default_graph()

    # Initialize the sequence decoder (this now builds HM-RNN automatically)
    nsd = nsd_module.NeuralSequenceDecoder(args)

    # Run inference
    out = nsd.inference()

    # LM-based decoding
    decoder_out = lmDecoderUtils.cer_with_lm_decoder(
        ngramDecoder, out, outputType='speech_sil', blankPenalty=np.log(2)
    )

    # Helper to convert ASCII arrays to text
    def _ascii_to_text(text):
        endIdx = np.argwhere(text==0)
        return ''.join([chr(char) for char in text[0:endIdx[0,0]]])

    for x in range(out['transcriptions'].shape[0]):
        trueTranscriptions[dirIdx].append(_ascii_to_text(out['transcriptions'][x,:]))
    decodedTranscriptions[dirIdx] = decoder_out['decoded_transcripts']


In [ ]:
# ============================================
#  Evaluation metrics (WER / CER)
# ============================================
from neuralDecoder.utils.lmDecoderUtils import _cer_and_wer as cer_and_wer

cer, wer = cer_and_wer(decodedTranscriptions[0], trueTranscriptions[0],
                       outputType='speech_sil', returnCI=True)

print("Word error rate (WER):", wer)
print("Phoneme/character error rate (CER):", cer)


In [ ]:
# ============================================
#  Print predictions and write submission file
# ============================================
print("\nDecoded sentences (test):")
print(decodedTranscriptions[0])

print("\nDecoded sentences (competition holdout):")
print(decodedTranscriptions[1])

# ✅ same as before
with open('baselineCompetitionSubmission_hmrnn.txt', 'w') as f:
    for x in range(len(decodedTranscriptions[1])):
        f.write(decodedTranscriptions[1][x]+'\n')

print("✅ Saved predictions to baselineCompetitionSubmission_hmrnn.txt")
